# Tutorial: `add_gap_entries` with MIMIC-IV ICD Demo Data

This notebook walks through how to use the `add_gap_entries` utility from `src/icd_gap_utils.py`.

The function inserts synthetic `GAP_<n>` token rows between consecutive ICD procedure entries whenever the time gap between them exceeds a threshold. This is useful for sequence modelling where temporal distances between billing codes need to be encoded explicitly.

**Topics covered:**
1. Load and inspect the demo dataset
2. Understand the columns relevant to gap insertion
3. Run `add_gap_entries` on a single admission
4. Run it across all admissions
5. Inspect and verify the output

## 1. Setup — imports and path configuration

In [1]:
import sys
import pathlib

# Make the src module importable from within the notebooks/ directory
repo_root = pathlib.Path.cwd().parent  # biling-codes/
sys.path.insert(0, str(repo_root / "src"))

import pandas as pd
from icd_gap_utils import add_gap_entries

print("Import OK — repo root:", repo_root)

Import OK — repo root: c:\Users\user\Documents\daniyal\biling-codes


## 2. Load the demo dataset

The file `data/mimic_4_icd_demo.parquet` contains MIMIC-IV procedure ICD codes with one row per procedure per hospital admission.

In [2]:
data_path = repo_root / "data" / "mimic_4_icd_demo.parquet"
df = pd.read_parquet(data_path)

print(f"Shape: {df.shape}")
print(f"Admissions (hadm_id): {df['hadm_id'].nunique():,}")
print(f"Patients (subject_id): {df['subject_id'].nunique():,}")
df.head()

Shape: (757313, 14)
Admissions (hadm_id): 185,412
Patients (subject_id): 113,775


,subject_id,hadm_id,seq_num,chartdate,icd_code,icd_version,icd_code_mapped,pos,age_month,age_day,segment,Lo7,expired,expired_7
0,14546051,20000069,1,2131-11-10,0KQM0ZZ,10,0KQM0ZZ,1,400,12000,1,0,0,0
1,14546051,20000069,2,2131-11-10,10E0XZZ,10,10E0XZZ,2,400,12000,1,0,0,0
2,13074106,20000102,1,2135-10-25,7359,9,M2060,1,229,6871,1,0,0,0
3,13074106,20000102,2,2135-10-25,7309,9,10907ZC,2,229,6871,1,0,0,0
4,14990224,20000147,1,2121-08-30,02100Z9,10,02100Z9,1,872,26173,1,0,0,0


## 3. Inspect the columns

The columns that `add_gap_entries` reads and writes are:

| Column | Role |
|---|---|
| `hadm_id` | Groups rows into a single hospital admission |
| `chartdate` | Date of the procedure (must be `datetime64`) |
| `seq_num` | Sequence number within the admission (secondary sort key) |
| `icd_code` | ICD procedure code — set to `GAP_<n>` for synthetic rows |
| `icd_code_mapped` | Mapped code — also set to `GAP_<n>` for synthetic rows |
| `pos` | Position indicator — set to `1` for synthetic rows |
| `icd_version` | ICD version — set to `"gap"` for synthetic rows |

All other columns are copied from the *preceding* real row when a gap row is created.

In [3]:
print(df.dtypes)
print("\nSample chartdate values:", df['chartdate'].head(3).tolist())

subject_id                  int64
hadm_id                     int64
seq_num                     int64
chartdate          datetime64[ns]
icd_code                   object
icd_version                object
icd_code_mapped            object
pos                         int64
age_month                   int64
age_day                     int64
segment                     int64
Lo7                         int64
expired                     int64
expired_7                   int64
dtype: object

Sample chartdate values: [Timestamp('2131-11-10 00:00:00'), Timestamp('2131-11-10 00:00:00'), Timestamp('2135-10-25 00:00:00')]


## 4. Pick a single admission with multiple procedure dates

To best demonstrate gap insertion we need an admission where procedures were charted on several different dates.  
Admission `20000235` has procedures spread across three dates (~5 and ~8 days apart), making it a clean example.

In [4]:
DEMO_HADM_ID = 20000235

demo = df[df['hadm_id'] == DEMO_HADM_ID].sort_values(['chartdate', 'seq_num'])
print(f"Rows before gap insertion: {len(demo)}")
print(f"Distinct chartdates: {demo['chartdate'].nunique()}")
demo[['hadm_id', 'chartdate', 'seq_num', 'icd_code', 'icd_code_mapped', 'icd_version']]

Rows before gap insertion: 4
Distinct chartdates: 3


,hadm_id,chartdate,seq_num,icd_code,icd_code_mapped,icd_version
12,20000235,2139-11-14,4,3995,5A1D70Z,9
13,20000235,2139-11-19,1,3723,4A020N8,9
14,20000235,2139-11-19,2,8856,B2000ZZ,9
15,20000235,2139-11-27,3,4523,0DJD8ZZ,9


## 5. Run `add_gap_entries` on a single admission

Pass `hadm_id_val=[DEMO_HADM_ID]` to restrict processing to one admission.  
The default `gap_list=[30, 10, 5, 1]` is used here, meaning gaps are filled greedily with the largest fitting token first.

In [5]:
demo_with_gaps = add_gap_entries(
    df,
    hadm_id_val=[DEMO_HADM_ID],
    gap_list=[30, 10, 5, 1],
)

print(f"Rows after gap insertion:  {len(demo_with_gaps)}")
print(f"Gap rows added:            {len(demo_with_gaps) - len(demo)}")
demo_with_gaps[['hadm_id', 'chartdate', 'seq_num', 'icd_code', 'icd_code_mapped', 'icd_version']]

Rows after gap insertion:  11
Gap rows added:            7


,hadm_id,chartdate,seq_num,icd_code,icd_code_mapped,icd_version
0,20000235,2139-11-14,4,3995,5A1D70Z,9
0,20000235,2139-11-15,4,GAP_1,GAP_1,gap
0,20000235,2139-11-16,4,GAP_1,GAP_1,gap
0,20000235,2139-11-17,4,GAP_1,GAP_1,gap
0,20000235,2139-11-18,4,GAP_1,GAP_1,gap
1,20000235,2139-11-19,1,3723,4A020N8,9
2,20000235,2139-11-19,2,8856,B2000ZZ,9
2,20000235,2139-11-24,2,GAP_5,GAP_5,gap
2,20000235,2139-11-25,2,GAP_1,GAP_1,gap
2,20000235,2139-11-26,2,GAP_1,GAP_1,gap


## 6. Verify the gap logic

Check that consecutive gap tokens correctly decompose each time interval.  
For a 5-day gap between `2139-11-14` and `2139-11-19`, a single `GAP_5` token should be inserted.

In [6]:
# Show only the gap rows to confirm their chartdates and tokens
gap_rows = demo_with_gaps[demo_with_gaps['icd_version'] == 'gap']
print(f"Total gap rows: {len(gap_rows)}")
gap_rows[['chartdate', 'icd_code', 'icd_code_mapped', 'icd_version', 'pos']]

Total gap rows: 7


,chartdate,icd_code,icd_code_mapped,icd_version,pos
0,2139-11-15,GAP_1,GAP_1,gap,1
0,2139-11-16,GAP_1,GAP_1,gap,1
0,2139-11-17,GAP_1,GAP_1,gap,1
0,2139-11-18,GAP_1,GAP_1,gap,1
2,2139-11-24,GAP_5,GAP_5,gap,1
2,2139-11-25,GAP_1,GAP_1,gap,1
2,2139-11-26,GAP_1,GAP_1,gap,1


In [8]:
# Consecutive date differences should equal the gap token value (in days)
dates = demo_with_gaps['chartdate'].tolist()
codes = demo_with_gaps['icd_code_mapped'].tolist()

print(f"{'chartdate':<15}  {'icd_code':<20}  {'days_since_prev':>15}")
print("-" * 55)
for i, (date, code) in enumerate(zip(dates, codes)):
    delta = (date - dates[i - 1]).days if i > 0 else 0
    print(f"{str(date.date()):<15}  {code:<20}  {delta:>15}")

chartdate        icd_code              days_since_prev
-------------------------------------------------------
2139-11-14       5A1D70Z                             0
2139-11-15       GAP_1                               1
2139-11-16       GAP_1                               1
2139-11-17       GAP_1                               1
2139-11-18       GAP_1                               1
2139-11-19       4A020N8                             1
2139-11-19       B2000ZZ                             0
2139-11-24       GAP_5                               5
2139-11-25       GAP_1                               1
2139-11-26       GAP_1                               1
2139-11-27       0DJD8ZZ                             1


## 7. Custom gap thresholds

You can pass any descending list to `gap_list`. For example, use finer-grained thresholds `[7, 3, 1]` for shorter stays.

In [9]:
demo_fine = add_gap_entries(
    df,
    hadm_id_val=[DEMO_HADM_ID],
    gap_list=[7, 3, 1],
)

print(f"Rows with gap_list=[7,3,1]: {len(demo_fine)}")
demo_fine[['chartdate', 'icd_code', 'icd_version']]

Rows with gap_list=[7,3,1]: 7


,chartdate,icd_code,icd_version
0,2139-11-14,3995,9
0,2139-11-17,GAP_3,gap
0,2139-11-18,GAP_1,gap
1,2139-11-19,3723,9
2,2139-11-19,8856,9
2,2139-11-26,GAP_7,gap
3,2139-11-27,4523,9


## 8. Process all admissions

Omit `hadm_id_val` (or pass `None`) to process every admission in the DataFrame.  
This may take a few minutes on the full dataset.

In [10]:
# Use a small slice (first 500 admissions) for a quick demo run
sample_hadm_ids = df['hadm_id'].unique()[:500].tolist()
df_sample = df[df['hadm_id'].isin(sample_hadm_ids)]

df_sample_gaps = add_gap_entries(df_sample)   # hadm_id_val=None → all admissions

n_real = len(df_sample)
n_total = len(df_sample_gaps)
n_gap = n_total - n_real

print(f"Admissions processed : {len(sample_hadm_ids)}")
print(f"Original rows        : {n_real:,}")
print(f"Gap rows inserted    : {n_gap:,}")
print(f"Total rows           : {n_total:,}")

# Distribution of gap token types
print("\nGap token frequency:")
print(df_sample_gaps[df_sample_gaps['icd_version'] == 'gap']['icd_code'].value_counts())

Admissions processed : 500
Original rows        : 2,017
Gap rows inserted    : 622
Total rows           : 2,639

Gap token frequency:
icd_code
GAP_1     554
GAP_5      46
GAP_10     20
GAP_30      2
Name: count, dtype: int64


## 9. Save the enriched dataset

Write the result to a new parquet file so it can be used downstream (e.g. for tokenisation or model training).

In [11]:
out_path = repo_root / "data" / "mimic_4_icd_demo_with_gaps.parquet"
df_sample_gaps.to_parquet(out_path, index=False)
print(f"Saved to: {out_path}")

Saved to: c:\Users\user\Documents\daniyal\biling-codes\data\mimic_4_icd_demo_with_gaps.parquet
